In [6]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from braindecode.models import EEGNet
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, cohen_kappa_score
from pathlib import Path

DATA_DIR = Path('data')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Use: {device}')

Use: cpu


In [4]:

# X shape: (5184, 22, 1001) → 5184 trials, 22 channels, 1001 timepoints
# y shape: (5184,)          → labels 0,1,2,3

d = np.load(DATA_DIR / 'bciv2a.npz', allow_pickle=True)
X = d['X'].astype(np.float32)
y = d['y'].astype(np.int64)

print(f'X: {X.shape}')
print(f'y: {y.shape}, classes: {np.unique(y)}')

X: (5184, 22, 1001)
y: (5184,), classes: [0 1 2 3]


In [11]:
# Import EEGNetv4 from braindecode
# in_chans              → số channels EEG (22 cho BCI IV 2a)
# n_classes             → 4 class (left/right/feet/tongue)
# input_window_samples  → số timepoints (1001)

def make_model(n_chans, n_outputs, n_times):
    model = EEGNet(
        n_chans=n_chans,
        n_outputs=n_outputs,
        n_times=n_times,
        drop_prob=0.5
    )
    return model.to(device)

model = make_model(n_chans=22, n_outputs=4, n_times=1001)
print(model)

Layer (type (var_name):depth-idx)                            Input Shape               Output Shape              Param #                   Kernel Shape
EEGNet (EEGNet)                                              [1, 22, 1001]             [1, 4]                    --                        --
├─Ensure4d (ensuredims): 1-1                                 [1, 22, 1001]             [1, 22, 1001, 1]          --                        --
├─Rearrange (dimshuffle): 1-2                                [1, 22, 1001, 1]          [1, 1, 22, 1001]          --                        --
├─Conv2d (conv_temporal): 1-3                                [1, 1, 22, 1001]          [1, 8, 22, 1002]          512                       [1, 64]
├─BatchNorm2d (bnorm_temporal): 1-4                          [1, 8, 22, 1002]          [1, 8, 22, 1002]          16                        --
├─ParametrizedConv2dWithConstraint (conv_spatial): 1-5       [1, 8, 22, 1002]          [1, 16, 1, 1002]          --                  

In [12]:
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [13]:
# Duyệt qua từng batch, tính loss, cập nhật weights
# zero_grad → xóa gradient cũ trước mỗi batch
# backward  → tính gradient
# step      → cập nhật weights

def train_epoch(loader):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [14]:
# Chạy model trên test set, không cập nhật weights
# argmax → lấy class có xác suất cao nhất

def evaluate(loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            preds   += model(X_batch.to(device)).argmax(dim=1).cpu().tolist()
            targets += y_batch.tolist()
    return accuracy_score(targets, preds), cohen_kappa_score(targets, preds)

In [21]:
EPOCHS, BATCH = 300, 64
kfold   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

In [23]:
# Mỗi fold: train 100 epoch → evaluate → ghi kết quả

for fold, (train_idx, test_idx) in enumerate(kfold.split(X, y)):
    train_loader = DataLoader(TensorDataset(torch.tensor(X[train_idx]), torch.tensor(y[train_idx])), batch_size=BATCH, shuffle=True)
    test_loader  = DataLoader(TensorDataset(torch.tensor(X[test_idx]),  torch.tensor(y[test_idx])),  batch_size=BATCH)

    model     = make_model(n_chans=22, n_outputs=4, n_times=1001)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(EPOCHS):
        loss = train_epoch(train_loader)

    acc, kappa = evaluate(test_loader)
    results.append((fold+1, acc, kappa))
    print(f'Fold {fold+1} → Acc: {acc:.4f} | Kappa: {kappa:.4f}')

Fold 1 → Acc: 0.6162 | Kappa: 0.4884
Fold 2 → Acc: 0.6181 | Kappa: 0.4908
Fold 3 → Acc: 0.6027 | Kappa: 0.4703
Fold 4 → Acc: 0.6095 | Kappa: 0.4793
Fold 5 → Acc: 0.5830 | Kappa: 0.4440


In [24]:
accs   = [r[1] for r in results]
kappas = [r[2] for r in results]
print(f'Trung bình → Acc: {np.mean(accs):.4f} ± {np.std(accs):.4f} | Kappa: {np.mean(kappas):.4f}')

Trung bình → Acc: 0.6059 ± 0.0127 | Kappa: 0.4746


In [25]:
# Train lại trên 100% data để save làm pretrained

full_loader = DataLoader(TensorDataset(torch.tensor(X), torch.tensor(y)), batch_size=BATCH, shuffle=True)
model       = make_model(n_chans=22, n_outputs=4, n_times=1001)
optimizer   = torch.optim.Adam(model.parameters(), lr=0.001)

In [26]:
for epoch in range(EPOCHS):
    loss = train_epoch(full_loader)
    if (epoch+1) % 20 == 0:
        print(f'Epoch {epoch+1} | Loss: {loss:.4f}')

Epoch 20 | Loss: 1.0669
Epoch 40 | Loss: 1.0316
Epoch 60 | Loss: 0.9853
Epoch 80 | Loss: 0.9658
Epoch 100 | Loss: 0.9421
Epoch 120 | Loss: 0.9345
Epoch 140 | Loss: 0.9363
Epoch 160 | Loss: 0.9310
Epoch 180 | Loss: 0.9202
Epoch 200 | Loss: 0.9143
Epoch 220 | Loss: 0.9087
Epoch 240 | Loss: 0.9082
Epoch 260 | Loss: 0.9016
Epoch 280 | Loss: 0.8984
Epoch 300 | Loss: 0.8919


In [27]:
torch.save(model.state_dict(), 'data/eegnet_pretrained_300.pth')
print('Saved: data/eegnet_pretrained.pth')

Saved: data/eegnet_pretrained.pth


In [28]:
import scipy.io
import os

# Xem thử 1 file
mat = scipy.io.loadmat('MIMED/Motor Imagery/Left Hand Up-Down Imagine/P01.mat')
print(mat.keys())

# Xem shape của từng key
for k, v in mat.items():
    if not k.startswith('_'):
        print(f'{k}: {type(v)}, {v.shape if hasattr(v, "shape") else v}')

dict_keys(['__header__', '__version__', '__globals__', 'Fs', 'channel', 'joined_data', 'subject'])
Fs: <class 'numpy.ndarray'>, (1, 1)
channel: <class 'numpy.ndarray'>, (1, 14)
joined_data: <class 'numpy.ndarray'>, (1, 4)
subject: <class 'numpy.ndarray'>, (1,)


In [29]:
# Xem bên trong joined_data
for i in range(4):
    arr = mat['joined_data'][0, i]
    print(f'Activity {i}: shape={arr.shape}')

# Xem channel names
print(mat['channel'])
print(f'Fs: {mat["Fs"][0,0]}')

Activity 0: shape=(2432, 14)
Activity 1: shape=(2432, 14)
Activity 2: shape=(2432, 14)
Activity 3: shape=(2432, 14)
[[array(['AF3'], dtype='<U3') array(['F7'], dtype='<U2')
  array(['F3'], dtype='<U2') array(['FC5'], dtype='<U3')
  array(['T7'], dtype='<U2') array(['P7'], dtype='<U2')
  array(['O1'], dtype='<U2') array(['O2'], dtype='<U2')
  array(['P8'], dtype='<U2') array(['T8'], dtype='<U2')
  array(['FC6'], dtype='<U3') array(['F4'], dtype='<U2')
  array(['F8'], dtype='<U2') array(['AF4'], dtype='<U3')]]
Fs: 128
